In [ ]:
# M6: FAILURE TWIN SIMULATOR
# Monte Carlo simulation that answers:
#   "What happens if we delay maintenance by X days?"
#   "What if we reduce load by Y%?"
#   "When is the optimal repair window?"

import pandas as pd
import numpy as np
import json
from snowflake.snowpark.context import get_active_session

session = get_active_session()
session.sql("USE DATABASE FAILURE_GENOME_DB").collect()
session.sql("USE SCHEMA ML_MODELS").collect()
print("Session ready.")

In [ ]:
session.sql("""
CREATE OR REPLACE PROCEDURE FAILURE_GENOME_DB.ML_MODELS.SIMULATE_FAILURE_TWIN(
    p_asset_id VARCHAR,
    p_current_vib_mag FLOAT,
    p_current_temp FLOAT,
    p_degradation_velocity FLOAT,
    p_current_rul_hours FLOAT,
    p_load_reduction_pct FLOAT,
    p_maintenance_delay_days FLOAT,
    p_rpm_adjustment_pct FLOAT
)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'numpy')
HANDLER = 'run'
EXECUTE AS CALLER
AS
$$
import numpy as np
import json

def run(session, p_asset_id, p_current_vib_mag, p_current_temp,
        p_degradation_velocity, p_current_rul_hours,
        p_load_reduction_pct, p_maintenance_delay_days, p_rpm_adjustment_pct):

    np.random.seed(42)
    N_SIMS = 1000
    HORIZON_DAYS = 30
    HOURS_PER_DAY = 24

    # Base degradation rate (mm/s per hour)
    base_rate = float(p_degradation_velocity) / HOURS_PER_DAY if p_degradation_velocity else 0.01

    # Adjust degradation rate based on scenario parameters
    load_factor = 1.0 - (float(p_load_reduction_pct) / 100.0) * 0.6
    rpm_factor = 1.0 + (float(p_rpm_adjustment_pct) / 100.0) * 0.8
    adjusted_rate = base_rate * load_factor * rpm_factor

    # Failure threshold (ISO 10816 unacceptable = 18 mm/s)
    FAILURE_THRESHOLD = 18.0
    current_vib = float(p_current_vib_mag)

    # Monte Carlo simulation
    failure_times = []
    daily_failure_probs = [0.0] * HORIZON_DAYS
    daily_vib_mean = [0.0] * HORIZON_DAYS
    daily_vib_p10 = [0.0] * HORIZON_DAYS
    daily_vib_p90 = [0.0] * HORIZON_DAYS

    all_trajectories = np.zeros((N_SIMS, HORIZON_DAYS))

    for sim in range(N_SIMS):
        vib = current_vib
        failed = False
        for day in range(HORIZON_DAYS):
            # Stochastic degradation: base rate + noise + occasional spikes
            daily_noise = np.random.normal(0, adjusted_rate * 0.3)
            spike = np.random.exponential(adjusted_rate * 2) if np.random.random() < 0.05 else 0
            vib += (adjusted_rate * HOURS_PER_DAY) + daily_noise + spike
            vib = max(vib, current_vib * 0.8)  # can't go much below current

            all_trajectories[sim, day] = vib

            if vib >= FAILURE_THRESHOLD and not failed:
                failure_times.append(day)
                failed = True

        if not failed:
            failure_times.append(HORIZON_DAYS + 1)

    failure_times = np.array(failure_times)

    # Compute daily statistics
    for day in range(HORIZON_DAYS):
        daily_vib_mean[day] = round(float(np.mean(all_trajectories[:, day])), 2)
        daily_vib_p10[day] = round(float(np.percentile(all_trajectories[:, day], 10)), 2)
        daily_vib_p90[day] = round(float(np.percentile(all_trajectories[:, day], 90)), 2)
        daily_failure_probs[day] = round(float(np.mean(failure_times <= day)), 4)

    # RUL distribution statistics
    median_rul = float(np.median(failure_times)) * HOURS_PER_DAY
    p10_rul = float(np.percentile(failure_times, 10)) * HOURS_PER_DAY
    p90_rul = float(np.percentile(failure_times, 90)) * HOURS_PER_DAY

    # Optimal intervention window
    # Find the day where P(failure) crosses 20% — repair before this
    intervention_day = HORIZON_DAYS
    for day in range(HORIZON_DAYS):
        if daily_failure_probs[day] >= 0.20:
            intervention_day = day
            break

    # Cost analysis
    REPAIR_COST = 2500.0
    FAILURE_COST_PER_HOUR = 12000.0
    AVG_DOWNTIME_HOURS = 48.0

    cost_if_repair_now = REPAIR_COST
    cost_if_failure = FAILURE_COST_PER_HOUR * AVG_DOWNTIME_HOURS + REPAIR_COST * 2.5

    # Delay cost: probability of failure during delay * failure cost
    delay_days = int(float(p_maintenance_delay_days))
    prob_failure_during_delay = daily_failure_probs[min(delay_days, HORIZON_DAYS - 1)] if delay_days > 0 else 0
    expected_cost_if_delayed = prob_failure_during_delay * cost_if_failure + (1 - prob_failure_during_delay) * REPAIR_COST

    result = {
        "asset_id": str(p_asset_id),
        "scenario": {
            "load_reduction_pct": float(p_load_reduction_pct),
            "maintenance_delay_days": float(p_maintenance_delay_days),
            "rpm_adjustment_pct": float(p_rpm_adjustment_pct)
        },
        "rul_distribution": {
            "median_hours": round(median_rul, 1),
            "p10_hours": round(p10_rul, 1),
            "p90_hours": round(p90_rul, 1),
            "simulations": N_SIMS
        },
        "failure_probability_by_day": daily_failure_probs,
        "vibration_forecast": {
            "mean": daily_vib_mean,
            "p10": daily_vib_p10,
            "p90": daily_vib_p90
        },
        "intervention_window": {
            "recommended_day": intervention_day,
            "recommended_hours": intervention_day * HOURS_PER_DAY,
            "rationale": f"P(failure) reaches 20% at day {intervention_day}"
        },
        "cost_analysis": {
            "repair_now_cost": cost_if_repair_now,
            "expected_failure_cost": round(cost_if_failure, 0),
            "expected_cost_if_delayed": round(expected_cost_if_delayed, 0),
            "savings_from_early_action": round(cost_if_failure - cost_if_repair_now, 0),
            "delay_risk_multiplier": round(expected_cost_if_delayed / cost_if_repair_now, 2) if cost_if_repair_now > 0 else 0
        }
    }

    return result
$$
""").collect()
print("SIMULATE_FAILURE_TWIN procedure created.")

In [ ]:
result = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.SIMULATE_FAILURE_TWIN(
    'ASSET_001',   -- asset_id
    12.5,          -- current vibration (high)
    78.0,          -- current temperature
    0.15,          -- degradation velocity (mm/s per day)
    72.0,          -- current predicted RUL
    0.0,           -- load reduction: 0% (no change)
    0.0,           -- maintenance delay: 0 days
    0.0            -- RPM adjustment: 0%
)
""").collect()

baseline = json.loads(result[0][0]) if isinstance(result[0][0], str) else result[0][0]
print("="*70)
print("SCENARIO 1: BASELINE (no intervention)")
print("="*70)
print(f"  Median RUL: {baseline['rul_distribution']['median_hours']}h")
print(f"  RUL Range (P10-P90): {baseline['rul_distribution']['p10_hours']}h - {baseline['rul_distribution']['p90_hours']}h")
print(f"  Recommended intervention: Day {baseline['intervention_window']['recommended_day']}")
print(f"  Repair now cost: ${baseline['cost_analysis']['repair_now_cost']:,.0f}")
print(f"  Expected failure cost: ${baseline['cost_analysis']['expected_failure_cost']:,.0f}")
print(f"  Savings from early action: ${baseline['cost_analysis']['savings_from_early_action']:,.0f}")

In [ ]:
result2 = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.SIMULATE_FAILURE_TWIN(
    'ASSET_001',
    12.5,
    78.0,
    0.15,
    72.0,
    30.0,          -- reduce load by 30%
    0.0,
    0.0
)
""").collect()

load_reduced = json.loads(result2[0][0]) if isinstance(result2[0][0], str) else result2[0][0]
print("="*70)
print("SCENARIO 2: LOAD REDUCTION 30%")
print("="*70)
print(f"  Median RUL: {load_reduced['rul_distribution']['median_hours']}h")
print(f"  RUL Range: {load_reduced['rul_distribution']['p10_hours']}h - {load_reduced['rul_distribution']['p90_hours']}h")
print(f"  Recommended intervention: Day {load_reduced['intervention_window']['recommended_day']}")

# Compare
rul_gain = load_reduced['rul_distribution']['median_hours'] - baseline['rul_distribution']['median_hours']
print(f"\n  >> RUL GAIN from 30% load reduction: +{rul_gain:.0f} hours ({rul_gain/24:.1f} days)")

In [ ]:
result3 = session.sql("""
CALL FAILURE_GENOME_DB.ML_MODELS.SIMULATE_FAILURE_TWIN(
    'ASSET_001',
    12.5,
    78.0,
    0.15,
    72.0,
    0.0,
    7.0,           -- delay maintenance by 7 days
    0.0
)
""").collect()

delayed = json.loads(result3[0][0]) if isinstance(result3[0][0], str) else result3[0][0]
print("="*70)
print("SCENARIO 3: MAINTENANCE DELAYED 7 DAYS")
print("="*70)
print(f"  P(failure during delay): {delayed['failure_probability_by_day'][6]*100:.1f}%")
print(f"  Expected cost if delayed: ${delayed['cost_analysis']['expected_cost_if_delayed']:,.0f}")
print(f"  Delay risk multiplier: {delayed['cost_analysis']['delay_risk_multiplier']}x")
print(f"\n  >> COST INCREASE from 7-day delay: ${delayed['cost_analysis']['expected_cost_if_delayed'] - baseline['cost_analysis']['repair_now_cost']:,.0f}")

In [ ]:
print("="*70)
print("SCENARIO COMPARISON — ASSET_001 (Compressor A1)")
print("="*70)
print(f"{'Scenario':<30} {'Median RUL':>12} {'Intervention':>14} {'Cost':>12}")
print("-"*70)
print(f"{'Baseline (no action)':<30} {baseline['rul_distribution']['median_hours']:>10.0f}h {'Day '+str(baseline['intervention_window']['recommended_day']):>14} ${baseline['cost_analysis']['repair_now_cost']:>10,.0f}")
print(f"{'Load reduction 30%':<30} {load_reduced['rul_distribution']['median_hours']:>10.0f}h {'Day '+str(load_reduced['intervention_window']['recommended_day']):>14} ${load_reduced['cost_analysis']['repair_now_cost']:>10,.0f}")
print(f"{'Delay 7 days':<30} {delayed['rul_distribution']['median_hours']:>10.0f}h {'Day '+str(delayed['intervention_window']['recommended_day']):>14} ${delayed['cost_analysis']['expected_cost_if_delayed']:>10,.0f}")
print(f"{'If failure occurs':<30} {'N/A':>12} {'N/A':>14} ${baseline['cost_analysis']['expected_failure_cost']:>10,.0f}")
print("-"*70)
print(f"\nRECOMMENDATION: Repair by Day {baseline['intervention_window']['recommended_day']} saves ${baseline['cost_analysis']['savings_from_early_action']:,.0f}")

In [ ]:
print("="*70)
print("M6: FAILURE TWIN SIMULATOR - COMPLETE")
print("="*70)
print("""
Architecture:
  - Monte Carlo simulation (1000 trajectories, 30-day horizon)
  - Stochastic degradation with random spikes (5% daily probability)
  - What-if scenarios: load reduction, maintenance delay, RPM adjustment
  - Cost-optimized intervention window

Outputs (all returned as JSON for dashboard consumption):
  - RUL distribution (median, P10, P90)
  - Daily failure probability curve (30 data points)
  - Vibration forecast with confidence bands (mean, P10, P90)
  - Optimal intervention day (when P(failure) crosses 20%)
  - Cost comparison: repair now vs delay vs failure

Dashboard integration:
  - failure_probability_by_day -> line chart (probability over time)
  - vibration_forecast -> area chart with bands (uncertainty visualization)
  - cost_analysis -> bar chart (compare scenarios)
  - Sliders: load_reduction, maintenance_delay, rpm_adjustment -> re-run simulation
""")